In [ ]:
import json
import os

import mlflow
from datasets import load_dataset
from dotenv import load_dotenv
# from langchain_core.rate_limiters import InMemoryRateLimiter
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from mlflow.genai.datasets import create_dataset
from mlflow.genai.scorers import Safety

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "dataset_tracking_example"

In [ ]:
# Warning - This step may take a while to download the dataset - The dataset size is ~40GB
wikipedia_dataset = load_dataset("wikimedia/wikipedia", "20231101.en")
wikipedia_dataset

In [ ]:
KEYWORD = "artificial intelligence"


# Filter for articles where the specific keyword if it appears at least 3 times
def filter_article_by_keyword(article):
    KEYWORD = "artificial intelligence"
    text_lower = article["text"].lower()
    count = text_lower.count(KEYWORD)
    return count >= 3


filtered_articles_dataset = wikipedia_dataset["train"].filter(
    filter_article_by_keyword,
    desc="Filtering articles with at least 3 occurrences of the keyword",
    batch_size=1000,
    writer_batch_size=1000,
    num_proc=4,
)

filtered_articles_dataset

In [ ]:
filtered_articles_df = filtered_articles_dataset.to_pandas()
filtered_articles_id_title_json = filtered_articles_df[["id", "title"]].to_json(
    orient="records"
)

# rate_limiter = InMemoryRateLimiter(
#     requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
#     check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
#     max_bucket_size=10,  # Controls the maximum burst size.
# )

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     rate_limiter=rate_limiter,
# )


llm = ChatOpenAI(
    model="gpt-5-nano",
)

prompt = f"""Your are provided with the id and title of the wikipedia articles. Identify the top 100 articles which matches best to the Topic - {KEYWORD} and return the ids of the articles in a list format.
Make sure to return only 100 ids in a json list format.
{filtered_articles_id_title_json}
"""

predicted_ids = llm.invoke(prompt)
predicted_ids


In [ ]:
predicted_ids_list = json.loads(
    predicted_ids.content.replace("```json", "").replace("```", "")
)
len(predicted_ids_list)

In [ ]:
filtered_articles_df = filtered_articles_df[
    filtered_articles_df["id"].isin(predicted_ids_list)
]
print(filtered_articles_df.shape)
filtered_articles_df.head()

In [ ]:
dataset = mlflow.data.from_pandas(
    df=filtered_articles_df, name=f"top_100_articles_{KEYWORD.replace(' ', '_')}"
)

with mlflow.start_run():
    mlflow.log_input(dataset, context="raw_data")

In [ ]:
data_list = []

for record in filtered_articles_df["text"].tolist():
    data_list.append({"inputs": {"text": record}, "expectations": {}})
data_list[0]

In [ ]:
summary_dataset = create_dataset(
    name="summarization_dataset",
    tags={"type": "summary", "source": "wikipedia"},
)

In [ ]:
summary_dataset.merge_records(data_list[:3])

In [ ]:
# from mlflow.genai.datasets import get_dataset
# get_dataset(dataset_id=summary_dataset.dataset_id)

In [ ]:
def predict_fn(text) -> str:
    prompt = f"""Summarize the text below as a bullet point list of the most important points.
    Text: {text}"""
    response = llm.invoke(prompt)
    print(type(response.content))
    return response.content


# 3.Run the evaluation
results = mlflow.genai.evaluate(
    data=summary_dataset, predict_fn=predict_fn, scorers=[Safety()]
)

In [ ]:
results.result_df

In [ ]:
annotated_dataset = create_dataset(
    name="annotated_dataset",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [ ]:
annotated_data_list = []
for index, row in results.result_df[["request", "response"]].iterrows():
    annotated_data_list.append(
        {
            "inputs": {"text": row["request"]["text"]},
            "expectations": {"response": row["response"]},
        }
    )

annotated_data_list[0]

In [ ]:
annotated_dataset.merge_records(annotated_data_list)

In [ ]:
from mlflow.genai.datasets import delete_dataset

delete_dataset(dataset_id=annotated_dataset.dataset_id)

In [ ]:
traces = mlflow.search_traces(max_results=10, return_type="list", run_id=results.run_id)
traces

In [ ]:
annotated_data_list = []
for trace in traces:
    trace_id = trace.to_dict()["info"]["trace_id"]
    request = trace.data._get_root_span().inputs["text"]
    response = trace.data._get_root_span().outputs
    annotated_data_list.append(
        {"inputs": {"text": request}, "expectations": {"response": response}}
    )

annotated_data_list[0]

In [ ]:
annotated_dataset_from_trace = create_dataset(
    name="annotated_dataset_from_trace",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [ ]:
annotated_dataset_from_trace.merge_records(annotated_data_list)